In [32]:
from sympy import Eq, symbols, solve, diff, lambdify

x, y, t, dt, dx, dy, v = symbols('x y t dt dx dy v')
u = symbols('u', cls=Function)

2D diffusion equation

In [20]:
continuous = Eq(diff(u(t, x, y), t), v*(diff(u(t, x, y), x, x) + diff(u(t, x, y), y, y)))
continuous

Eq(Derivative(u(t, x, y), t), v*(Derivative(u(t, x, y), (x, 2)) + Derivative(u(t, x, y), (y, 2))))

Discrestise the equation with a central difference scheme. Take the taylor expasion of the neighboring cells around the point u(t, x, y). Terms after the third are ignored as they are negligibly small.

In [23]:
x1 =  u(t, x, y) + dx*diff(u(t, x, y), x) + ((dx*dx)/2)*diff(u(t, x, y), x, x) + ((dx*dx*dx)/3)*diff(u(t, x, y), x, x, x)
x2 =  u(t, x, y) - dx*diff(u(t, x, y), x) + ((dx*dx)/2)*diff(u(t, x, y), x, x) - ((dx*dx*dx)/3)*diff(u(t, x, y), x, x, x)

central_diff = Eq(u(t, x+1, y) + u(t, x-1, y), x1 + x2)
central_diff

Eq(u(t, x - 1, y) + u(t, x + 1, y), dx**2*Derivative(u(t, x, y), (x, 2)) + 2*u(t, x, y))

Solve the equation for the second order derivative.

In [26]:
derivative = solve(central_diff, diff(u(t, x, y), x, x))[0]
derivative

(-2*u(t, x, y) + u(t, x - 1, y) + u(t, x + 1, y))/dx**2

Substitue this expression back into the continuous formula for x and y. Discretise the derivative of time using the definition of a derivative.

In [30]:
discrete = Eq((u(t+1, x, y) - u(t, x, y))/dt, v*((-2*u(t, x, y) + u(t, x-1, y) + u(t, x+1, y))/(dx*dx) + (-2*u(t, x, y) + u(t, x, y-1) + u(t, x, y+1))/(dy*dy)))
discrete

Eq((-u(t, x, y) + u(t + 1, x, y))/dt, v*((-2*u(t, x, y) + u(t, x, y - 1) + u(t, x, y + 1))/dy**2 + (-2*u(t, x, y) + u(t, x - 1, y) + u(t, x + 1, y))/dx**2))

Solve for u(t+1, x, y) to allow the advancement of time.

In [33]:
solution = solve(discrete, u(t+1, x, y))[0]
update = lambdify([dx, dy, dt, v, u(t, x, y), u(t, x+1, y), u(t, x-1, y), u(t, x, y+1), u(t, x, y-1)], solution)
solution

(dt*dx**2*v*(-2*u(t, x, y) + u(t, x, y - 1) + u(t, x, y + 1)) + dt*dy**2*v*(-2*u(t, x, y) + u(t, x - 1, y) + u(t, x + 1, y)) + dx**2*dy**2*u(t, x, y))/(dx**2*dy**2)